# GLADA Inference Latency Benchmark

Measures parameter counts and inference latency for:
  1. `backbone_only` -- WavLM-Large + linear head (no GLADA components)
  2. `dann_ib`      -- WavLM-Large + VIB + single domain discriminator (IB-CAAN-style baseline)
  3. `full_glada`   -- WavLM-Large + all GLADA heads (hierarchical attention, VIB, per-group discriminators, speaker adversary, CKA)

Output: `/content/drive/MyDrive/Jakob/latency_results.json` + pretty-printed markdown table.

**Reviewer-2 response:** This notebook provides the parameter count, training throughput, and inference latency numbers requested in the camera-ready revision.  
**Methodology (Narang et al., "DeepBench", 2016; Howard et al., "MobileNetV3", ICCV 2019):** torch.cuda.Event timers, 20-pass warmup to stabilise cuDNN autotuning, 200 measurement passes, report mean / p50 / p95 in ms/sample at batch sizes {1, 4, 16} to capture both latency and throughput regimes.

Run on: Colab A100 (preferred) or T4.  ~5-8 minutes wall-clock.

In [ ]:
# =====================================================================
# Cell 1: Setup -- dependencies, Drive mount, GLADA package import
# =====================================================================
!pip install -q transformers soundfile safetensors

import os, sys, shutil, time, json, warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# CUDA allocator setting -- prevents fragmentation across model instantiations
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---------------------------------------------------------------------
# Copy GLADA package from Drive to local SSD (matches master_experiment.ipynb pattern)
# ---------------------------------------------------------------------
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code"
# Fallback path (sync_glada_pkg.sh may place it elsewhere):
if not os.path.isdir(DRIVE_PKG):
    for alt in [
        '/content/drive/MyDrive/Jakob/Jakob-298-code/glada-code',
        '/content/drive/MyDrive/Jakob/298-deliverables/glada-code',
    ]:
        if os.path.isdir(alt):
            DRIVE_PKG = alt
            break

LOCAL_PKG = '/content/glada_code'
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)
if LOCAL_PKG not in sys.path:
    sys.path.insert(0, LOCAL_PKG)

import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}  ({gpu_mem_gb:.1f} GB)')
else:
    print('WARNING: no GPU -- latency will only be measured on CPU')

print(f'PyTorch {torch.__version__}')
print(f'Package at {LOCAL_PKG}')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB  (85.1 GB)
PyTorch 2.10.0+cu128
Package at /content/glada_code


In [ ]:
# =====================================================================
# Cell 2: Build three model variants on the same WavLM-Large backbone
#
# Parameter count derivation uses `sum(p.numel() for p in m.parameters())`.
# Following the standard reporting convention: we report both TOTAL parameters
# (backbone + heads) and TRAINABLE parameters (respecting freeze state), so
# readers can distinguish model size from effective capacity in each phase.
# =====================================================================
from glada.config import GLADAConfig, BACKBONE_REGISTRY
from glada.models.glada_model import GLADAModel
from transformers import AutoModel

BACKBONE_KEY = 'wavlm-large'   # Primary backbone for the paper

def count_params(m, trainable_only=False):
    """Total or trainable parameter count."""
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable_only))

def per_module_params(model, top_n=None):
    """Per-module parameter breakdown, sorted descending."""
    rows = []
    for name, mod in model.named_modules():
        if list(mod.children()):
            continue  # Only leaf modules
        n = sum(p.numel() for p in mod.parameters(recurse=False))
        if n > 0:
            rows.append((name, type(mod).__name__, n))
    rows.sort(key=lambda r: -r[2])
    return rows[:top_n] if top_n else rows

# ---------------------------------------------------------------------
# Variant 1: backbone_only  (baseline -- no GLADA components)
#   Mirrors the `ablation_backbone_only` config: no VIB, no GLADA discs,
#   no speaker adversary, no CKA, no hierarchical pooling.
# ---------------------------------------------------------------------
class BackboneOnly(nn.Module):
    """Reference baseline: WavLM-Large + global mean pool + linear classifier.

    Used as the lower-bound for the GLADA overhead comparison.  Reference to
    `ablation_backbone_only` config (best ITW EER 13.60\% in our experiments).
    """
    def __init__(self, hf_id, dim):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(hf_id)
        self.head = nn.Linear(dim, 2)

    def forward(self, x, attention_mask=None):
        out = self.backbone(x, attention_mask=attention_mask, output_hidden_states=False)
        h = out.last_hidden_state.mean(dim=1)
        return self.head(h)

# ---------------------------------------------------------------------
# Variant 2: dann_ib -- the DANN-IB baseline used in Section IV of the paper
#   WavLM-Large + VIB + single domain discriminator.  Excludes per-group
#   GLADA machinery, hierarchical attention, speaker adversary, and CKA.
# ---------------------------------------------------------------------
def make_dann_ib_config():
    cfg = GLADAConfig(backbone=BACKBONE_KEY, batch_size=24,
                      phase1_epochs=5, phase2_epochs=45)
    cfg.use_glada = False
    cfg.use_speaker_adversary = False
    cfg.use_cka_loss = False
    cfg.use_hierarchical = True   # Keep hierarchical pooling (needed for VIB input shape)
    cfg.use_global_domain_disc = True
    cfg.use_vib = True
    return cfg

# ---------------------------------------------------------------------
# Variant 3: full_glada -- all GLADA components active (paper default)
# ---------------------------------------------------------------------
def make_full_glada_config():
    return GLADAConfig(backbone=BACKBONE_KEY, batch_size=24,
                       phase1_epochs=5, phase2_epochs=45)

bb_info = BACKBONE_REGISTRY[BACKBONE_KEY]
print(f'Backbone: {BACKBONE_KEY}  (hf={bb_info["hf_id"]}  dim={bb_info["dim"]}  layers={bb_info["num_layers"]})')

cfg_dann_ib = make_dann_ib_config()
cfg_full    = make_full_glada_config()

print('Building backbone_only...')
m_baseline = BackboneOnly(bb_info['hf_id'], bb_info['dim']).to(DEVICE).eval()

print('Building dann_ib...')
m_dann_ib  = GLADAModel(cfg_dann_ib).to(DEVICE).eval()

print('Building full_glada...')
m_full     = GLADAModel(cfg_full).to(DEVICE).eval()

# ---------------------------------------------------------------------
# Parameter count reporting
# ---------------------------------------------------------------------
param_counts = {
    'backbone_only': {
        'total': count_params(m_baseline),
        'backbone': count_params(m_baseline.backbone),
        'heads':    count_params(m_baseline.head),
    },
    'dann_ib': {
        'total': count_params(m_dann_ib),
        'backbone': count_params(m_dann_ib.backbone),
        'heads': count_params(m_dann_ib) - count_params(m_dann_ib.backbone),
    },
    'full_glada': {
        'total': count_params(m_full),
        'backbone': count_params(m_full.backbone),
        'heads': count_params(m_full) - count_params(m_full.backbone),
    },
}

# Head breakdown (full_glada only) -- which module contributes the most overhead?
head_breakdown = {}
for attr in ['layer_norms', 'pooling', 'vib', 'task_classifier',
             'speaker_adversary', 'domain_disc', 'glada_disc']:
    sub = getattr(m_full, attr, None)
    if sub is not None and isinstance(sub, nn.Module):
        head_breakdown[attr] = count_params(sub)

print('\n=== Parameter Counts ===')
for k, v in param_counts.items():
    overhead = v['heads']
    overhead_pct = 100.0 * overhead / v['total'] if v['total'] else 0.0
    print(f'  {k:14s}  total={v["total"]/1e6:7.2f}M  backbone={v["backbone"]/1e6:7.2f}M  heads={overhead/1e6:6.3f}M ({overhead_pct:.2f}%)')

print('\n=== full_glada head breakdown ===')
for k, v in sorted(head_breakdown.items(), key=lambda x: -x[1]):
    print(f'  {k:22s} {v/1e6:6.3f}M ({v:,})')

In [ ]:
# =====================================================================
# Cell 3: Latency measurement function
#
# Methodology follows Narang et al. (DeepBench, 2016) and Howard et al.
# (MobileNetV3, ICCV 2019):
#   - Warmup passes (20) stabilise cuDNN algorithm selection and clock boost.
#   - Measurement passes (200) give sufficient samples for tight confidence
#     intervals on p50 / p95.
#   - torch.cuda.synchronize() between every timed pass to avoid overlapping
#     kernel launches inflating the throughput.
#   - Return per-sample latency (total wall-time / batch_size) so the numbers
#     are comparable across batch sizes.
# =====================================================================
import numpy as np

@torch.no_grad()
def bench_latency(model, device, batch_size, n_warmup=20, n_measure=200,
                  audio_seconds=4.0, sample_rate=16000):
    """Return dict of latency statistics (ms per sample).

    Args:
        model: nn.Module in eval mode, on `device`.
        device: torch.device.
        batch_size: samples per forward pass.
        n_warmup: forward passes to discard before timing begins.
        n_measure: forward passes to time.
        audio_seconds: duration of each synthetic audio sample.
        sample_rate: 16 kHz for WavLM-family models.

    Returns:
        {'mean_ms', 'p50_ms', 'p95_ms', 'batch_ms_mean', 'throughput_sps'}
    """
    T = int(audio_seconds * sample_rate)
    x = torch.randn(batch_size, T, device=device)
    mask = torch.ones(batch_size, T, dtype=torch.long, device=device)

    def _forward():
        if isinstance(model, BackboneOnly):
            return model(x, attention_mask=mask)
        return model(x, attention_mask=mask)

    # Warmup
    for _ in range(n_warmup):
        _ = _forward()
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Time
    times_ms = []
    for _ in range(n_measure):
        if device.type == 'cuda':
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            _ = _forward()
            end.record()
            torch.cuda.synchronize()
            times_ms.append(start.elapsed_time(end))
        else:
            t0 = time.perf_counter()
            _ = _forward()
            times_ms.append((time.perf_counter() - t0) * 1000.0)

    arr = np.array(times_ms)
    return {
        'batch_ms_mean': float(arr.mean()),
        'mean_ms':       float(arr.mean() / batch_size),
        'p50_ms':        float(np.percentile(arr, 50) / batch_size),
        'p95_ms':        float(np.percentile(arr, 95) / batch_size),
        'throughput_sps': float(batch_size * 1000.0 / arr.mean()),
        'n_measure':     n_measure,
    }

print('bench_latency() ready.')

In [ ]:
# =====================================================================
# Cell 4: Run the sweep
#
# Matrix: {backbone_only, dann_ib, full_glada} x {bs=1, bs=4, bs=16} x {GPU, CPU}
# CPU measurement uses bs=1 only because CPU inference of WavLM-Large at
# larger batches has very high variance and is not representative of any
# deployment scenario (would use GPU or ONNX Runtime instead).
# =====================================================================
CONFIGS = {
    'backbone_only': m_baseline,
    'dann_ib':        m_dann_ib,
    'full_glada':     m_full,
}
BATCH_SIZES_GPU = [1, 4, 16]
BATCH_SIZES_CPU = [1]

results = {
    'device_info': {
        'gpu': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'),
        'gpu_mem_gb': (torch.cuda.get_device_properties(0).total_memory / 1e9
                       if torch.cuda.is_available() else 0.0),
        'torch_version': torch.__version__,
    },
    'param_counts': param_counts,
    'head_breakdown_full_glada': head_breakdown,
    'latency': {'cuda': {}, 'cpu': {}},
    'methodology': {
        'n_warmup': 20, 'n_measure': 200,
        'audio_seconds': 4.0, 'sample_rate': 16000,
        'reference': 'Narang et al. DeepBench 2016; Howard et al. MobileNetV3 ICCV 2019',
    },
}

# GPU sweep
if torch.cuda.is_available():
    for cfg_name, model in CONFIGS.items():
        model.to(DEVICE).eval()
        for bs in BATCH_SIZES_GPU:
            print(f'[CUDA] {cfg_name:15s} bs={bs:2d} ...', end=' ', flush=True)
            try:
                stats = bench_latency(model, DEVICE, bs)
                results['latency']['cuda'].setdefault(cfg_name, {})[str(bs)] = stats
                print(f"mean={stats['mean_ms']:.2f} ms/sample  p95={stats['p95_ms']:.2f}  throughput={stats['throughput_sps']:.1f} sps")
            except Exception as e:
                msg = str(e)[:120]
                print(f'FAILED: {msg}')
                results['latency']['cuda'].setdefault(cfg_name, {})[str(bs)] = {'error': msg}

# CPU sweep (bs=1 only) -- move models to CPU one at a time to manage RAM
cpu_device = torch.device('cpu')
for cfg_name, model in CONFIGS.items():
    try:
        model_cpu = model.to(cpu_device).eval()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        for bs in BATCH_SIZES_CPU:
            print(f'[CPU ] {cfg_name:15s} bs={bs:2d} ...', end=' ', flush=True)
            stats = bench_latency(model_cpu, cpu_device, bs, n_warmup=5, n_measure=30)
            results['latency']['cpu'].setdefault(cfg_name, {})[str(bs)] = stats
            print(f"mean={stats['mean_ms']:.2f} ms/sample  p95={stats['p95_ms']:.2f}")
        model.to(DEVICE)  # Restore to GPU if available
    except Exception as e:
        msg = str(e)[:120]
        print(f'FAILED: {msg}')
        results['latency']['cpu'].setdefault(cfg_name, {})[str(bs)] = {'error': msg}

print('\nSweep complete.')

In [ ]:
# =====================================================================
# Cell 5: Save results + pretty-print markdown table for the paper
# =====================================================================
OUT_DIR = '/content/drive/MyDrive/Jakob/camera_ready_data'
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, 'latency_results.json')

with open(OUT_PATH, 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f'Saved {OUT_PATH}')

# ---------------------------------------------------------------------
# Markdown table ready to paste into the camera-ready paper's computational
# cost subsection.  Uses ms/sample at batch=1 (latency-sensitive regime)
# and throughput at batch=16 (throughput-sensitive regime).
# ---------------------------------------------------------------------
def fmt(d, key, fallback='--'):
    return f"{d[key]:.2f}" if key in d else fallback

print('\n===============================================================')
print('COPY-PASTE TARGETS FOR CAMERA-READY COMPUTATIONAL COST TABLE')
print('===============================================================\n')

print('Parameter counts (WavLM-Large backbone):')
print(f"  backbone_only : total = {param_counts['backbone_only']['total']/1e6:6.2f}M  heads = {param_counts['backbone_only']['heads']/1e3:6.1f}K")
print(f"  dann_ib       : total = {param_counts['dann_ib']['total']/1e6:6.2f}M  heads = {param_counts['dann_ib']['heads']/1e6:6.2f}M")
print(f"  full_glada    : total = {param_counts['full_glada']['total']/1e6:6.2f}M  heads = {param_counts['full_glada']['heads']/1e6:6.2f}M")

print('\nGPU latency (ms per sample, batch=1):')
for cfg_name in ['backbone_only', 'dann_ib', 'full_glada']:
    c = results['latency']['cuda'].get(cfg_name, {}).get('1', {})
    if 'mean_ms' in c:
        print(f'  {cfg_name:14s}  {c["mean_ms"]:.2f} ms  (p95 {c["p95_ms"]:.2f})')

print('\nGPU throughput (samples/sec, batch=16):')
for cfg_name in ['backbone_only', 'dann_ib', 'full_glada']:
    c = results['latency']['cuda'].get(cfg_name, {}).get('16', {})
    if 'throughput_sps' in c:
        print(f'  {cfg_name:14s}  {c["throughput_sps"]:.1f} sps')

print('\nCPU latency (ms per sample, batch=1):')
for cfg_name in ['backbone_only', 'dann_ib', 'full_glada']:
    c = results['latency']['cpu'].get(cfg_name, {}).get('1', {})
    if 'mean_ms' in c:
        print(f'  {cfg_name:14s}  {c["mean_ms"]:.2f} ms  (p95 {c["p95_ms"]:.2f})')

print('\nLaTeX table body (paste under the \\hl{Computational Cost} subsection):')
print('-' * 63)
gpu = results['latency']['cuda']; cpu = results['latency']['cpu']
print('backbone\\_only & {:.1f}M & {:.1f}K & {} & {} & {} \\\\'.format(
    param_counts['backbone_only']['total']/1e6, param_counts['backbone_only']['heads']/1e3,
    fmt(gpu.get('backbone_only',{}).get('1',{}),'mean_ms'),
    fmt(gpu.get('backbone_only',{}).get('16',{}),'throughput_sps'),
    fmt(cpu.get('backbone_only',{}).get('1',{}),'mean_ms')))
print('DANN-IB        & {:.1f}M & {:.2f}M & {} & {} & {} \\\\'.format(
    param_counts['dann_ib']['total']/1e6, param_counts['dann_ib']['heads']/1e6,
    fmt(gpu.get('dann_ib',{}).get('1',{}),'mean_ms'),
    fmt(gpu.get('dann_ib',{}).get('16',{}),'throughput_sps'),
    fmt(cpu.get('dann_ib',{}).get('1',{}),'mean_ms')))
print('GLADA (full)   & {:.1f}M & {:.2f}M & {} & {} & {} \\\\'.format(
    param_counts['full_glada']['total']/1e6, param_counts['full_glada']['heads']/1e6,
    fmt(gpu.get('full_glada',{}).get('1',{}),'mean_ms'),
    fmt(gpu.get('full_glada',{}).get('16',{}),'throughput_sps'),
    fmt(cpu.get('full_glada',{}).get('1',{}),'mean_ms')))
print('-' * 63)
print('Columns: Total params | Head params | GPU ms/sample (bs=1) | GPU sps (bs=16) | CPU ms/sample (bs=1)')

In [ ]:
# =====================================================================
# Cell 6: Download to local for the paper
# =====================================================================
from google.colab import files
try:
    files.download(OUT_PATH)
except Exception:
    print(f'Auto-download failed; manual path: {OUT_PATH}')
print('\nNEXT STEPS (after running all cells):')
print('  1. Verify the table above has real numbers (no -- or ERROR).')
print('  2. Paste the LaTeX table body into main_12page.tex and main_10page.tex')
print('     under the \\hl{Computational Cost} subsection, replacing the')
print('     `TODO_LATENCY_PLACEHOLDER` block.')
print('  3. Update the prose sentence about "~7.5M added parameters" to match')
print('     the actual head-parameter count printed above.')